# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm going with **Lane 2: Refresh / Content Opportunity Scoring**.

Here's my thinking — after running the starter pipeline and poking around the data in notebooks 01 and 02, the thing that stuck with me is how badly the hand-written baseline does at picking the right pages to review. It gets 12 out of 50 right. That's worse than a coin flip on a dataset where 54% of pages are already declining. But the random forest gets 34 right — same data, same features, just a smarter way of combining them.

That gap tells me there's real signal buried in these columns that a flat rule can't capture. The refresh scoring lane lets me dig into that: build a better ranked queue, attach reason codes so a reviewer knows *why* each page showed up, and check honestly whether the improvement holds on unseen clients. I also like that the output is concrete — it's a list someone could actually open on Monday morning and start working through, not just a report that says 'engagement matters.'

One more thing: 73% of pages in the dataset have at least 100 impressions, so most of the inventory has enough signal for the model to work with. That's not always true in SEO datasets — sometimes you're staring at a sea of zeros and hoping for miracles.

In [1]:
# Quick sanity check: how much of the inventory has enough signal to work with?
import pandas as pd, numpy as np
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f'Total pages: {len(df):,}')
print(f'Clients: {df["client_id"].nunique()}')
print(f'Pages with >= 100 impressions: {(df["impressions_90d"] >= 100).sum():,} '
      f'({(df["impressions_90d"] >= 100).mean()*100:.1f}%)')
print(f'Declining pages (trend_direction == down): {(df["trend_direction"]=="down").sum():,} '
      f'({(df["trend_direction"]=="down").mean()*100:.1f}%)')


Total pages: 30,000Clients: 32Pages with >= 100 impressions: 22,006 (73.4%)Declining pages (trend_direction == down): 16,262 (54.2%)

## 2. The question: decision, action, cost of a wrong call

**Decision:** which pages should a content team review first when they have limited bandwidth? They can't look at all 30,000 pages (or 500k+ in the full warehouse). They need a short, ranked list.

**Who acts:** a content editor or SEO strategist at FlyRank's client company. They open the list, start at the top, and decide for each page: refresh it, rewrite the title/meta, expand the content, merge it with a duplicate, or leave it alone.

**Cost of a wrong recommendation:**
- **False positive (flagged but fine):** wasted reviewer time. An editor spends 15-30 minutes reading a page, checking search console, deciding it doesn't need work. Multiply that by the length of the queue and it adds up fast.
- **False negative (missed a real decline):** a page that needed attention keeps losing traffic quietly. The client doesn't notice until the quarterly review, by which point the ranking loss is harder to reverse.

The false-positive cost is real but recoverable (just time). The false-negative cost compounds — so precision at the top of the list matters most, but recall still matters for the full picture.

**Why ML helps:** the starter baseline uses four hand-tuned weights (visibility 40%, freshness 30%, position opportunity 25%, depth gap 5%). That's a decent starting point, but it can't adapt to the actual patterns in the data. A page might have moderate impressions and a slightly stale update date but a very specific combination of position, CTR, and engagement that screams 'this one is slipping.' A flat rule misses that. A model that sees all the signals together can learn which combinations actually predict decline — and the pipeline already showed it does: ~2.8x better at Precision@50.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Three numbers that make this lane feel worth pursuing:

In [3]:
# What does the baseline actually get right vs wrong?
import json
res = json.load(open('outputs/model_results.json'))
base_p50 = res['baseline']['baseline_precision_at_50']
rf_p50 = res['models']['random_forest']['precision_at_50']
print(f'Baseline picks {round(base_p50*50)} actually-declining pages in its top 50 (Precision@50 = {base_p50:.3f})')
print(f'Random Forest picks {round(rf_p50*50)} actually-declining pages in its top 50 (Precision@50 = {rf_p50:.3f})')
print(f'Lift: {rf_p50/base_p50:.1f}x — the model finds roughly {round(rf_p50*50) - round(base_p50*50)} '
      f'more real problems in the same-sized review queue.')


## 4. Careful words: what I can and can't claim

**What I can say:**
- The model *observes* patterns in historical search and content signals that are associated with declining traffic.
- Pages ranked higher by the model are more likely to be declining than pages ranked by the hand-written rule — measured on held-out clients the model never trained on.
- The ranked queue is a *decision-support tool*: it helps a reviewer spend limited time on the most promising candidates first.
- The reason codes explain *which signals* contributed to a page's score, not *why* it declined.

**What I cannot say:**
- I cannot claim that refreshing a flagged page will *cause* it to recover. That would need a controlled experiment — an A/B test where some flagged pages get refreshed and others don't, with enough volume and time to measure the difference.
- I cannot claim to have discovered how Google's ranking algorithm works. The model finds associations in observable data; those associations could reflect algorithm behavior, user behavior, content quality, seasonality, or all of the above.
- I cannot claim the model generalizes beyond these 32 clients. It might — but I'd need a broader dataset to test that.

**The honest frame:**
> For a content strategist deciding which pages to review first, I'll build a ranked refresh queue from observable search and content signals, scored by a model trained on client-holdout validation, measured by Precision@K. A wrong call costs wasted reviewer time (false positive) or a missed decline (false negative). A plain rule isn't enough because the signals interact in ways a flat weighted score can't capture. I'll claim only observed, directional, decision-support results.

In [4]:
# Number 1: declining rate by content type — not all content declines equally
ct = df.groupby('content_type').agg(
    n=('content_id', 'count'),
    decline_rate=('trend_direction', lambda x: (x=='down').mean())
).sort_values('decline_rate', ascending=False)
print('Decline rate by content type:')
for t, row in ct.iterrows():
    print(f'  {t}: {row["decline_rate"]*100:.1f}% declining (n={int(row["n"]):,})')
print()

# Number 2: pages on page 1 with low CTR — immediate opportunity
page1_low_ctr = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10) &
                   (df['impressions_90d'] >= 500) & (df['ctr'] < 0.5)]
print(f'Pages on page 1 (pos 1-10) with 500+ impressions but CTR below 0.5%: '
      f'{len(page1_low_ctr):,}')
print('These are visible pages where a title/meta tweak might capture more clicks.')
print()

# Number 3: freshness tier vs decline — recently updated content declines more than stale
ft = df.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    decline_rate=('trend_direction', lambda x: (x=='down').mean())
).sort_values('decline_rate', ascending=False)
print('Decline rate by freshness tier:')
for t, row in ft.iterrows():
    print(f'  {t}: {row["decline_rate"]*100:.1f}% declining (n={int(row["n"]):,})')
print()
print('Interesting: the 91-180 day tier has the highest decline rate (61.1%), not the oldest.')
print('A simple "stale = bad" rule would miss this — which is exactly why a model might help.')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.